<a href="https://colab.research.google.com/github/Miranics/TradesREvamp/blob/main/research/02_strategy_bakeoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Strategy bake-off: gold, 2024 to 2026

Tests six intraday gold strategies and two trend-following variants on the same 1-minute data, with the same costs, year by year. The sweep model was designed on 2026 data, so **2024 and 2025 are its out-of-sample test**.

Run the cells top to bottom (Runtime → Run all works). Price data comes from **Exness's own free tick archive** (XAUUSDm, the symbol you trade), converted to 1-minute candles with the real spread. The first run saves it to `My Trades/data` in your Drive; later runs reuse it. Costs use Exness's actual median New York spread. Copy everything printed from Step 6 onwards back to the chat.

| Code | Strategy | Source of the idea |
|---|---|---|
| A1, A2 | Sweep model: London range → NY sweep → 5-min displacement → FVG entry (hold, or with adds) | Your ICT model, tested in this project |
| B | NY intraday momentum: first 30 min direction, hold to 16:00 ET | Peer-reviewed intraday momentum studies |
| C | NY opening-range breakout: first 5-min candle, stop at its other side, 10R | Zarattini & Aziz, plus an independent replication |
| D1, D2 | Session drift: always sell NY hours / always buy Asian hours | World Gold Council 2026 session data |
| E | Trend following on daily closes, 20 and 60 days | Time-series momentum research |

**Step 1.** Connect Google Drive and find the `My Trades` folder (approve the Drive pop-up).

In [1]:
import glob
import os

from google.colab import drive

drive.mount("/content/drive")

matches = [p for p in glob.glob("/content/drive/MyDrive/*") if os.path.basename(p).strip() == "My Trades"]
if not matches:
    raise FileNotFoundError("Could not find a folder called 'My Trades' at the top of My Drive")
BASE = matches[0]
DATA = os.path.join(BASE, "data")
OUT = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("Data folder:   ", DATA)
print("Results folder:", OUT)


Mounted at /content/drive
Data folder:    /content/drive/MyDrive/My Trades /data
Results folder: /content/drive/MyDrive/My Trades /results


**Step 2.** Download Exness XAUUSDm tick data month by month (3 at a time) and save 1-minute candles to Drive. About 10–20 minutes the first time. The current month isn't published until it ends, so it shows 404.

In [2]:
import io
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import pandas as pd
import requests

START, END = date(2024, 1, 1), date(2026, 10, 1)
BASE_URL = "https://ticks.ex2archive.com/ticks"


def months(start, end):
    y, m = start.year, start.month
    while date(y, m, 1) <= end:
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)


def ticks_to_minutes(raw):
    """Exness ticks (Timestamp, Bid, Ask) -> 1-minute bid candles plus the average spread."""
    cols = {c.lower(): c for c in raw.columns}
    t = pd.to_datetime(raw[cols["timestamp"]], utc=True, format="mixed")
    bid, ask = raw[cols["bid"]].astype(float), raw[cols["ask"]].astype(float)
    df = pd.DataFrame({"bid": bid.to_numpy(), "spread": (ask - bid).to_numpy()}, index=t).sort_index()
    m = df["bid"].resample("1min").ohlc().dropna()
    m["spread"] = df["spread"].resample("1min").mean().reindex(m.index)
    m.insert(0, "timestamp", (m.index - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta("1ms"))
    return m


def fetch_month(symbol, prefix, y, m):
    name = f"{prefix}_{y}_{m:02d}"
    out = os.path.join(DATA, name + ".csv")
    current = (y, m) == (date.today().year, date.today().month)
    if os.path.exists(out) and os.path.getsize(out) > 100_000 and not current:
        return name, "already in Drive"
    url = f"{BASE_URL}/{symbol}/{y}/{m:02d}/Exness_{symbol}_{y}_{m:02d}.zip"
    err = ""
    for attempt in range(1, 4):
        try:
            r = requests.get(url, timeout=300)
            if r.status_code == 404:
                return name, "not published yet (404)"
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
                with z.open(csv_name) as f:
                    raw = pd.read_csv(f, usecols=lambda c: c.lower() in ("timestamp", "bid", "ask"))
            bars = ticks_to_minutes(raw)
            bars.to_csv(out, index=False)
            return name, f"saved ({len(bars):,} minutes, avg spread {bars['spread'].mean():.3f})"
        except Exception as e:
            err = f"{type(e).__name__}: {e}"[:200]
            time.sleep(10 * attempt)
    return name, f"FAILED -> {err}"


def download(symbol, prefix, workers=3):
    jobs = list(months(START, END))
    with ThreadPoolExecutor(workers) as ex:
        for name, status in ex.map(lambda ym: fetch_month(symbol, prefix, *ym), jobs):
            print(f"{name}: {status}")


download("XAUUSDm", "xauusd")


xauusd_2024_01: saved (30,161 minutes, avg spread 0.219)
xauusd_2024_02: saved (28,699 minutes, avg spread 0.215)
xauusd_2024_03: saved (27,531 minutes, avg spread 0.218)
xauusd_2024_04: saved (27,973 minutes, avg spread 0.211)
xauusd_2024_05: saved (31,262 minutes, avg spread 0.215)
xauusd_2024_06: saved (27,412 minutes, avg spread 0.203)
xauusd_2024_07: saved (31,428 minutes, avg spread 0.189)
xauusd_2024_08: saved (30,159 minutes, avg spread 0.173)
xauusd_2024_09: saved (28,804 minutes, avg spread 0.160)
xauusd_2024_10: saved (31,655 minutes, avg spread 0.160)
xauusd_2024_11: saved (28,347 minutes, avg spread 0.160)
xauusd_2024_12: saved (28,496 minutes, avg spread 0.163)
xauusd_2025_01: saved (30,019 minutes, avg spread 0.161)
xauusd_2025_02: saved (27,351 minutes, avg spread 0.160)
xauusd_2025_03: saved (29,013 minutes, avg spread 0.160)
xauusd_2025_04: saved (28,847 minutes, avg spread 0.161)
xauusd_2025_05: saved (29,984 minutes, avg spread 0.160)
xauusd_2025_06: saved (28,780 m

**Step 3.** Load the gold data and check for missing months.

In [3]:
import glob
import os

import numpy as np
import pandas as pd

SPREAD = 0.30
SPREAD_HIGH = 0.60
OZ_PER_001_LOT = 1


def load_m1(pattern):
    files = sorted(glob.glob(os.path.join(DATA, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {DATA}")
    df = pd.concat(pd.read_csv(f) for f in files)
    df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.drop_duplicates("time").set_index("time").sort_index()
    keep = [c for c in ("open", "high", "low", "close", "spread") if c in df.columns]
    df = df[keep].astype(float)
    return df[df.index.dayofweek < 5]


m1 = load_m1("xauusd_*.csv")
m1_ny = m1.tz_convert("America/New_York")

if "spread" in m1.columns:
    ny_spread = m1.loc[(m1.index.hour >= 13) & (m1.index.hour < 21), "spread"].median()
    SPREAD = round(float(ny_spread), 2)
    SPREAD_HIGH = round(2 * SPREAD, 2)
print(f"Cost per position: ${SPREAD:.2f} (Exness median spread in New York hours); stress test: ${SPREAD_HIGH:.2f}")

per_month = m1.groupby(m1.index.strftime("%Y-%m")).size()
print(f"Gold minutes loaded: {len(m1):,}  from {m1.index[0]:%Y-%m-%d} to {m1.index[-1]:%Y-%m-%d}")
thin = per_month.iloc[:-1][per_month.iloc[:-1] < 15000]
if len(thin):
    print("Months with too little data (re-run Step 2 to fill them):", ", ".join(thin.index))
else:
    print("Every month looks complete.")


Cost per position: $0.16 (Exness median spread in New York hours); stress test: $0.32
Gold minutes loaded: 968,409  from 2024-01-01 to 2026-10-13
Every month looks complete.


**Step 4.** Load the trading logic (prints nothing). The sweep model code is identical to the earlier tests.

In [4]:
DISPLACEMENT = 1.5
BUFFER = 0.50


def bracket(d, entry, stop, path, rr=None):
    """Gross points for one position entered at `entry` on the first bar of `path`.
    Stop and target are checked on every bar; if both are touched in the same bar the stop wins."""
    risk = abs(entry - stop)
    if risk <= 0 or len(path) == 0:
        return None
    hi, lo = path["high"].to_numpy(), path["low"].to_numpy()
    n = len(path)
    sl = (lo <= stop) if d == 1 else (hi >= stop)
    i_sl = int(np.argmax(sl)) if sl.any() else n
    i_tp = n
    if rr is not None:
        target = entry + d * rr * risk
        tp = (hi >= target) if d == 1 else (lo <= target)
        i_tp = int(np.argmax(tp)) if tp.any() else n
    if i_tp < i_sl:
        return rr * risk
    if i_sl < n:
        return -risk
    return d * (path["close"].iloc[-1] - entry)


def swing_points(b, side):
    s = b["low"] if side == "low" else b["high"]
    if side == "low":
        return (s < s.shift(1)) & (s < s.shift(2)) & (s < s.shift(-1)) & (s < s.shift(-2))
    return (s > s.shift(1)) & (s > s.shift(2)) & (s > s.shift(-1)) & (s > s.shift(-2))


def find_setup(b, lh, ll):
    body = (b["close"] - b["open"]).abs()
    avg_body = body.rolling(20, min_periods=5).median().shift(1)
    ny = b[(b.index.hour >= 13) & (b.index.hour < 17)]
    for t in ny.index:
        i = b.index.get_loc(t)
        bar = b.iloc[i]
        for d, swept in ((-1, bar["high"] > lh), (1, bar["low"] < ll)):
            if not swept:
                continue
            prior = b.iloc[max(0, i - 40):i]
            sw = swing_points(b.iloc[max(0, i - 42):i + 1], "low" if d == -1 else "high")
            sw = sw.reindex(prior.index).fillna(False).astype(bool)
            if not sw.any():
                continue
            level = prior.loc[sw[sw].index[-1], "low" if d == -1 else "high"]
            extreme = bar["high"] if d == -1 else bar["low"]
            for j in range(i + 1, min(i + 30, len(b))):
                x = b.iloc[j]
                extreme = max(extreme, x["high"]) if d == -1 else min(extreme, x["low"])
                broke = x["close"] < level if d == -1 else x["close"] > level
                ab = avg_body.iloc[j]
                strong = body.iloc[j] >= DISPLACEMENT * (ab if ab and not np.isnan(ab) else np.inf)
                if broke and strong:
                    for k in range(j + 1, i, -1):
                        if k >= len(b) or k - 2 < 0:
                            continue
                        a, c = b.iloc[k - 2], b.iloc[k]
                        if d == -1 and a["low"] > c["high"]:
                            return dict(dir=d, entry=c["high"], stop=extreme + BUFFER,
                                        ready=b.index[min(k + 1, len(b) - 1)])
                        if d == 1 and a["high"] < c["low"]:
                            return dict(dir=d, entry=c["low"], stop=extreme - BUFFER,
                                        ready=b.index[min(k + 1, len(b) - 1)])
                    return None
            return None
    return None


def find_fill(s, d1):
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    after = d1[(d1.index >= s["ready"]) & (d1.index.hour < 21)]
    for t, x in after.iterrows():
        if (d == -1 and x["high"] >= stop) or (d == 1 and x["low"] <= stop):
            return None
        if (d == -1 and x["high"] >= entry) or (d == 1 and x["low"] <= entry):
            return None if t.hour >= 20 else t
    return None


def run_from_fill(d, entry, stop, d1, fill_t, adds=()):
    """Gross points and number of positions for a trade filled at fill_t, managed to 21:00 UTC."""
    risk = abs(entry - stop)
    window = d1[(d1.index >= fill_t) & (d1.index.hour < 21)]
    after = window[window.index > fill_t]
    units, todo = [entry], list(adds)
    for t, x in after.iterrows():
        hit_sl = x["high"] >= stop if d == -1 else x["low"] <= stop
        if hit_sl:
            return sum(d * (stop - e) for e in units), len(units)
        best = (entry - x["low"]) if d == -1 else (x["high"] - entry)
        while todo and best >= todo[0] * risk:
            units.append(entry + d * todo.pop(0) * risk)
            stop = entry
    last = window["close"].iloc[-1]
    return sum(d * (last - e) for e in units), len(units)


**Step 5.** Run every strategy over every day (a few minutes). Prints the number of trades per strategy per year.

In [5]:
rows = []


def add(strategy, day, d, gross, units, gross_opp, units_opp, risk=np.nan):
    if gross is None or gross_opp is None:
        return
    rows.append(dict(strategy=strategy, day=pd.Timestamp(day), dir=d, gross=gross, units=units,
                     gross_opp=gross_opp, units_opp=units_opp, risk=risk))


# A. Sweep model (London range -> NY sweep -> 5-min displacement break -> FVG limit entry)
for day, d1 in m1.groupby(m1.index.date):
    london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
    if len(london) < 200:
        continue
    b = d1.resample("5min").agg({"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
    s = find_setup(b, london["high"].max(), london["low"].min())
    if not s:
        continue
    fill_t = find_fill(s, d1)
    if fill_t is None:
        continue
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    opp_stop = 2 * entry - stop
    risk = abs(entry - stop)
    for name, adds in (("A1 Sweep model, hold to 23:00", ()), ("A2 Sweep model, adds at +1R/+2R", (1, 2))):
        g, u = run_from_fill(d, entry, stop, d1, fill_t, adds)
        go, uo = run_from_fill(-d, entry, opp_stop, d1, fill_t, adds)
        add(name, day, d, g, u, go, uo, risk)

# B, C, D. New York session strategies (times in New York time, so they follow US daylight saving)
for day, d1 in m1_ny.groupby(m1_ny.index.date):
    hm = d1.index.hour * 60 + d1.index.minute
    first30 = d1[(hm >= 570) & (hm < 600)]
    first5 = d1[(hm >= 570) & (hm < 575)]
    session = d1[(hm >= 600) & (hm < 960)]
    after_orb = d1[(hm >= 575) & (hm < 960)]
    if len(first30) < 25 or len(session) < 300:
        continue

    # B. Intraday momentum: direction of 09:30-10:00, held 10:00-16:00, no stop
    move = first30["close"].iloc[-1] - first30["open"].iloc[0]
    if move != 0:
        d = 1 if move > 0 else -1
        entry, last = session["open"].iloc[0], session["close"].iloc[-1]
        add("B  NY momentum (first 30 min -> hold to 16:00 ET)", day, d,
            d * (last - entry), 1, -d * (last - entry), 1)

    # C. Opening range breakout: direction of the first 5-min candle, stop at its other side, 10R or 16:00 ET
    o, c = first5["open"].iloc[0], first5["close"].iloc[-1]
    hi, lo = first5["high"].max(), first5["low"].min()
    if c != o and len(after_orb):
        d = 1 if c > o else -1
        entry = after_orb["open"].iloc[0]
        stop, opp_stop = (lo, hi) if d == 1 else (hi, lo)
        g = bracket(d, entry, stop, after_orb, rr=10)
        go = bracket(-d, entry, opp_stop, after_orb, rr=10)
        add("C  NY opening-range breakout (5-min, 10R)", day, d, g, 1, go, 1, abs(entry - stop))

    # D. Session drift: always short New York hours 09:30-16:00 ET
    ny = d1[(hm >= 570) & (hm < 960)]
    add("D1 Always SELL New York hours", day, -1, ny["open"].iloc[0] - ny["close"].iloc[-1], 1,
        ny["close"].iloc[-1] - ny["open"].iloc[0], 1)

# D2. Session drift: always long Asian hours 00:00-07:00 UTC
for day, d1 in m1.groupby(m1.index.date):
    asia = d1[d1.index.hour < 7]
    if len(asia) < 300:
        continue
    add("D2 Always BUY Asian hours", day, 1, asia["close"].iloc[-1] - asia["open"].iloc[0], 1,
        asia["open"].iloc[0] - asia["close"].iloc[-1], 1)

trades = pd.DataFrame(rows)
trades["year"] = trades["day"].dt.year
print(trades.groupby(["strategy", "year"]).size().unstack(fill_value=0).to_string())


year                                               2024  2025  2026
strategy                                                           
A1 Sweep model, hold to 23:00                        94    64    56
A2 Sweep model, adds at +1R/+2R                      94    64    56
B  NY momentum (first 30 min -> hold to 16:00 ET)   249   249   196
C  NY opening-range breakout (5-min, 10R)           249   249   196
D1 Always SELL New York hours                       249   249   196
D2 Always BUY Asian hours                           258   258   202


**Step 6.** Scoreboard by year: win rate, profit, profit factor, t-stat, and whether it beats random direction.

In [6]:
rng = np.random.default_rng(42)
N_RANDOM = 5000


def stats(net, net_opp):
    n = len(net)
    if n == 0:
        return {}
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    sd = net.std(ddof=1) if n > 1 else np.nan
    pick = rng.random((N_RANDOM, n)) < 0.5
    rand_tot = np.where(pick, net.to_numpy(), net_opp.to_numpy()).sum(axis=1)
    return {"trades": n,
            "win_%": round(100 * (net > 0).mean(), 1),
            "avg_$": round(net.mean(), 2),
            "total_$": round(net.sum(), 1),
            "profit_factor": round(wins / losses, 2) if losses > 0 else np.inf,
            "t_stat": round(net.mean() / sd * np.sqrt(n), 2) if sd and sd > 0 else np.nan,
            "p_vs_random": round((rand_tot >= net.sum()).mean(), 3)}


def scoreboard(spread, by=("strategy", "year")):
    t = trades.copy()
    t["net"] = t["gross"] - t["units"] * spread
    t["net_opp"] = t["gross_opp"] - t["units_opp"] * spread
    out = []
    for key, g in t.groupby(list(by)):
        out.append({**dict(zip(by, key if isinstance(key, tuple) else (key,))), **stats(g["net"], g["net_opp"])})
    for key, g in t.groupby("strategy"):
        row = {"strategy": key, **{b: "ALL" for b in by if b != "strategy"}}
        out.append({**row, **stats(g["net"], g["net_opp"])})
    return pd.DataFrame(out).sort_values(list(by), key=lambda s: s.astype(str)).reset_index(drop=True)


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("Profit is in $ per 0.01 lot (1 oz of gold). 'p_vs_random' = share of 5,000 random-direction")
print("versions of the same trades that did at least as well. Below 0.05 means the direction choice adds real value.\n")
print(f"## Scoreboard by year (spread ${SPREAD:.2f} per position)\n")
board = scoreboard(SPREAD)
print(board.to_string(index=False))


Profit is in $ per 0.01 lot (1 oz of gold). 'p_vs_random' = share of 5,000 random-direction
versions of the same trades that did at least as well. Below 0.05 means the direction choice adds real value.

## Scoreboard by year (spread $0.16 per position)

                                         strategy year  trades  win_%  avg_$  total_$  profit_factor  t_stat  p_vs_random
                    A1 Sweep model, hold to 23:00 2024      94   43.6   0.96     90.1           1.28    0.83        0.214
                    A1 Sweep model, hold to 23:00 2025      64   48.4   1.76    112.3           1.35    0.82        0.114
                    A1 Sweep model, hold to 23:00 2026      56   53.6  11.73    657.1           2.29    2.25        0.048
                    A1 Sweep model, hold to 23:00  ALL     214   47.7   4.02    859.5           1.75    2.50        0.012
                  A2 Sweep model, adds at +1R/+2R 2024      94   40.4   1.56    146.5           1.37    0.95        0.138
              

**Step 7.** Buys vs sells, the sweep model in R, and the same results with a doubled spread.

In [7]:
t = trades.copy()
t["net"] = t["gross"] - t["units"] * SPREAD
t["side"] = np.where(t["dir"] == 1, "buy", "sell")

print("## Buys vs sells, by year (total $ per 0.01 lot)\n")
two_sided = t[~t["strategy"].str.startswith("D")]
print(two_sided.pivot_table(index=["strategy", "side"], columns="year", values="net",
                            aggfunc="sum", margins=True, margins_name="ALL").round(1).to_string())

print("\n## Sweep model in R (1R = distance from entry to stop)\n")
sw = t[t["strategy"].str.startswith("A")].copy()
sw["R"] = sw["net"] / sw["risk"]
print(sw.groupby(["strategy", "year"]).agg(
    trades=("R", "size"), avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1)),
    median_stop_dollars=("risk", lambda x: round(x.median(), 2))).to_string())
print("\n2026 is the period the model was designed on. 2024 and 2025 are the true out-of-sample test.")

print(f"\n## Same scoreboard with a wider spread (${SPREAD_HIGH:.2f}), all years combined\n")
wide = scoreboard(SPREAD_HIGH)
print(wide[wide["year"] == "ALL"].drop(columns="year").to_string(index=False))


## Buys vs sells, by year (total $ per 0.01 lot)

year                                                     2024   2025    2026     ALL
strategy                                          side                              
A1 Sweep model, hold to 23:00                     buy    37.9   83.2   254.1   375.3
                                                  sell   52.2   29.1   403.0   484.3
A2 Sweep model, adds at +1R/+2R                   buy    68.6  197.8   334.7   601.0
                                                  sell   78.0   93.8   618.5   790.3
B  NY momentum (first 30 min -> hold to 16:00 ET) buy   -21.6  270.3  -317.2   -68.5
                                                  sell  -31.3 -285.1   469.8   153.4
C  NY opening-range breakout (5-min, 10R)         buy  -102.5  -75.1  -235.4  -413.0
                                                  sell  102.2  183.0   196.8   482.0
ALL                                                     183.4  497.0  1724.4  2404.8

## Sweep model

**Step 8.** Trend following on daily closes, compared with simply buying and holding.

In [8]:
day_key = (m1_ny.index - pd.Timedelta(hours=17)).date
daily = m1_ny["close"].groupby(day_key).last()
daily.index = pd.to_datetime(daily.index)
daily = daily[daily.index.dayofweek < 5]
chg = daily.diff()

tsm_rows = []
for lb in (20, 60):
    pos = np.sign(daily - daily.shift(lb)).shift(1)
    flips = pos.diff().abs().fillna(0) / 2
    pnl = (pos * chg - flips * SPREAD).dropna()
    for yr, g in pnl.groupby(pnl.index.year):
        tsm_rows.append({"strategy": f"E  Trend following, {lb}-day (daily, multi-day holds)", "year": yr,
                         "days": len(g), "position_changes": int(flips.loc[g.index].sum()),
                         "total_$": round(g.sum(), 1),
                         "t_stat": round(g.mean() / g.std(ddof=1) * np.sqrt(len(g)), 2)})
for yr, g in chg.dropna().groupby(chg.dropna().index.year):
    tsm_rows.append({"strategy": "Buy and hold (reference)", "year": yr, "days": len(g), "position_changes": 0,
                     "total_$": round(g.sum(), 1), "t_stat": round(g.mean() / g.std(ddof=1) * np.sqrt(len(g)), 2)})

print("## Trend following on daily closes (17:00 New York), $ per 0.01 lot\n")
print(pd.DataFrame(tsm_rows).to_string(index=False))
print("\nOvernight swap is not included. On Exness gold it can add up over multi-day holds.")


## Trend following on daily closes (17:00 New York), $ per 0.01 lot

                                           strategy  year  days  position_changes  total_$  t_stat
E  Trend following, 20-day (daily, multi-day holds)  2024   186                14    205.6    0.55
E  Trend following, 20-day (daily, multi-day holds)  2025   206                27    460.5    0.66
E  Trend following, 20-day (daily, multi-day holds)  2026   162                13   1474.3    1.29
E  Trend following, 60-day (daily, multi-day holds)  2024   146                 2    219.0    0.63
E  Trend following, 60-day (daily, multi-day holds)  2025   206                 6   1525.4    2.20
E  Trend following, 60-day (daily, multi-day holds)  2026   162                 4    201.8    0.18
                           Buy and hold (reference)  2024   206                 0    565.5    1.50
                           Buy and hold (reference)  2025   206                 0   1694.5    2.45
                           Buy and hold 

**Step 9.** Verdict: which strategies pass all four robustness checks. Also saves the results to `My Trades/results`.

In [9]:
verdict = []
for name, g in board[board["year"] != "ALL"].groupby("strategy"):
    allrow = board[(board["strategy"] == name) & (board["year"] == "ALL")].iloc[0]
    yearly = dict(zip(g["year"], g["total_$"]))
    every_year_positive = all(v > 0 for v in yearly.values())
    wide_total = wide[(wide["strategy"] == name) & (wide["year"] == "ALL")]["total_$"].iloc[0]
    checks = {"positive every year": every_year_positive,
              "t-stat above 2": allrow["t_stat"] > 2,
              "beats random direction (p < 0.05)": allrow["p_vs_random"] < 0.05,
              "still positive at wide spread": wide_total > 0}
    verdict.append(dict(strategy=name, **{str(y): v for y, v in yearly.items()},
                        checks_passed=f"{sum(checks.values())}/4",
                        failed=", ".join(k for k, v in checks.items() if not v) or "none"))

print("## Verdict: totals in $ per 0.01 lot, and which robustness checks each strategy passes\n")
print(pd.DataFrame(verdict).sort_values("checks_passed", ascending=False).to_string(index=False))
print("\nA strategy worth automating should pass all 4. D1 and D2 are fixed-direction rules, so")
print("'beats random direction' only means the market drifted that way in this sample.")

trades.to_csv(os.path.join(OUT, "bakeoff_trades.csv"), index=False)
board.to_csv(os.path.join(OUT, "bakeoff_scoreboard.csv"), index=False)
print(f"\nSaved bakeoff_trades.csv and bakeoff_scoreboard.csv to {OUT}")


## Verdict: totals in $ per 0.01 lot, and which robustness checks each strategy passes

                                         strategy  2024   2025  2026 checks_passed                                                                                                failed
                    A1 Sweep model, hold to 23:00  90.1  112.3 657.1           4/4                                                                                                  none
                  A2 Sweep model, adds at +1R/+2R 146.5  291.6 953.2           4/4                                                                                                  none
                    D1 Always SELL New York hours  -8.3 -299.3 659.1           1/4                                positive every year, t-stat above 2, beats random direction (p < 0.05)
                        D2 Always BUY Asian hours 139.7  465.5 -25.4           1/4                                positive every year, t-stat above 2, beats random directio

## Optional: dollar regime test
**Step 10.** Download Exness's dollar index (DXYm) the same way.

In [10]:
download("DXYm", "dollaridx")


dollaridx_2024_01: saved (31,366 minutes, avg spread 0.025)
dollaridx_2024_02: saved (30,164 minutes, avg spread 0.032)
dollaridx_2024_03: saved (29,963 minutes, avg spread 0.034)
dollaridx_2024_04: saved (29,991 minutes, avg spread 0.050)
dollaridx_2024_05: saved (32,787 minutes, avg spread 0.064)
dollaridx_2024_06: saved (28,901 minutes, avg spread 0.066)
dollaridx_2024_07: saved (33,026 minutes, avg spread 0.065)
dollaridx_2024_08: saved (31,454 minutes, avg spread 0.062)
dollaridx_2024_09: saved (30,301 minutes, avg spread 0.062)
dollaridx_2024_10: saved (33,059 minutes, avg spread 0.063)
dollaridx_2024_11: saved (29,949 minutes, avg spread 0.063)
dollaridx_2024_12: saved (29,994 minutes, avg spread 0.039)
dollaridx_2025_01: saved (31,567 minutes, avg spread 0.064)
dollaridx_2025_02: saved (28,749 minutes, avg spread 0.065)
dollaridx_2025_03: saved (30,136 minutes, avg spread 0.066)
dollaridx_2025_04: saved (31,278 minutes, avg spread 0.060)
dollaridx_2025_05: saved (31,216 minutes

**Step 11.** Does the sweep model do better when the dollar has a clear direction before 15:00 Kigali?

In [11]:
dxy = load_m1("dollaridx_*.csv")

dd = {}
for day, d in dxy.groupby(dxy.index.date):
    pre = d[d.index.hour < 13]
    if len(pre) < 300:
        continue
    dd[pd.Timestamp(day)] = dict(high=d["high"].max(), low=d["low"].min(), pre_high=pre["high"].max(),
                                 pre_low=pre["low"].min(), move=pre["close"].iloc[-1] - pre["open"].iloc[0])
dd = pd.DataFrame(dd).T.sort_index()
dd["prev_high"], dd["prev_low"] = dd["high"].shift(1), dd["low"].shift(1)
dd["strong"] = dd["move"].abs() > dd["move"].abs().rolling(40, min_periods=10).median().shift(1)
dd["breakout"] = (dd["pre_high"] > dd["prev_high"]) ^ (dd["pre_low"] < dd["prev_low"])

sw = trades[trades["strategy"].str.startswith("A")].copy()
sw["net"] = sw["gross"] - sw["units"] * SPREAD
sw["R"] = sw["net"] / sw["risk"]
sw = sw.join(dd[["move", "strong", "breakout"]], on="day").dropna(subset=["move"])
sw["dollar_breakout"] = sw["breakout"].astype(bool)
sw["dollar_strong_move"] = sw["strong"].astype(bool)
sw["gold_trade_against_dollar_move"] = sw["dir"] == -np.sign(sw["move"])
sw["against_dollar_and_strong"] = sw["gold_trade_against_dollar_move"] & sw["dollar_strong_move"]

print("Dollar signals are all measured before 15:00 Kigali (13:00 UTC), so they are known before the trade.\n")
for col in ["dollar_breakout", "dollar_strong_move", "gold_trade_against_dollar_move", "against_dollar_and_strong"]:
    print(f"## {col}\n")
    print(sw.groupby(["strategy", col]).agg(trades=("R", "size"), win_pct=("R", lambda x: round((x > 0).mean() * 100, 1)),
          avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1))).to_string(), "\n")


Dollar signals are all measured before 15:00 Kigali (13:00 UTC), so they are known before the trade.

## dollar_breakout

                                                 trades  win_pct  avg_R  total_R
strategy                        dollar_breakout                                 
A1 Sweep model, hold to 23:00   False                80     51.2  0.193     15.5
                                True                134     45.5  0.193     25.9
A2 Sweep model, adds at +1R/+2R False                80     42.5  0.302     24.2
                                True                134     39.6  0.499     66.9 

## dollar_strong_move

                                                    trades  win_pct  avg_R  total_R
strategy                        dollar_strong_move                                 
A1 Sweep model, hold to 23:00   False                  104     46.2  0.236     24.5
                                True                   110     49.1  0.153     16.8
A2 Sweep model, adds at +1R/+2R